In [ ]:
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tensorflow.keras.utils import to_categorical

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = Path("data")
MODEL_PATH = "models/hand_landmarker.task"

RAW_CSV = "raw_landmarks.csv"

VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}

N_HANDS = 2
N_LANDMARKS = 21

COLUMNS = ["frame", "hand", "landmark", "x", "y", "z"]

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


# ============================================================
# 1. RAW LANDMARK EXTRACTION
# ============================================================

def extract(video_path, model_path=MODEL_PATH):
    """
    Extract raw hand landmarks from one video.

    Returns:
        DataFrame containing:
        frame, hand, landmark, x, y, z, frame_count
    """

    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)),
        running_mode=RunningMode.VIDEO,
        num_hands=N_HANDS,
    )

    video = cv2.VideoCapture(str(video_path))

    if not video.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = video.get(cv2.CAP_PROP_FPS)

    if not np.isfinite(fps) or fps <= 0:
        fps = 30

    rows = []
    frame_number = 0

    try:
        with HandLandmarker.create_from_options(options) as detector:

            while True:
                success, frame = video.read()

                if not success:
                    break

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=rgb
                )

                timestamp = int(frame_number * 1000 / fps)

                result = detector.detect_for_video(
                    mp_image,
                    timestamp
                )

                for hand_index, hand in enumerate(result.hand_landmarks):

                    for landmark_id, point in enumerate(hand):

                        rows.append((
                            frame_number,
                            hand_index,
                            landmark_id,
                            point.x,
                            point.y,
                            point.z
                        ))

                frame_number += 1

    finally:
        video.release()

    landmarks = pd.DataFrame(rows, columns=COLUMNS)

    # Preserve original video length
    landmarks["frame_count"] = frame_number

    return landmarks


def build_raw_dataset(overwrite=False):
    """
    Extract landmarks from videos and save them to CSV.

    If raw_landmarks.csv already exists, only process new videos.
    Set overwrite=True to process every video again.
    """

    paths = sorted(
        p for p in DATA_DIR.rglob("*")
        if p.suffix.lower() in VIDEO_EXTS
    )

    print(f"Found {len(paths)} videos under {DATA_DIR}")

    # Load existing results if available
    if Path(RAW_CSV).exists() and not overwrite:

        existing_df = pd.read_csv(RAW_CSV)

        processed_ids = set(existing_df["clip_id"].unique())

        print(f"Loaded {len(processed_ids)} previously processed videos.")

    else:

        existing_df = None
        processed_ids = set()

    new_landmarks = []
    failed = []

    for i, path in enumerate(paths, 1):

        signer = path.parent.parent.name.replace("ASL_", "")
        label = path.parent.name.strip().lower()
        clip_id = f"{signer}_{path.stem}"

        # Skip videos already in the CSV
        if clip_id in processed_ids:
            continue

        try:

            landmarks = extract(path)

            if landmarks.empty:
                raise ValueError("No hands detected in video.")

            # Add metadata to every landmark
            landmarks["clip_id"] = clip_id
            landmarks["label"] = label
            landmarks["signer"] = signer
            landmarks["path"] = str(path)

            new_landmarks.append(landmarks)

            print(f"[{i}/{len(paths)}] Processed {clip_id}")

        except (RuntimeError, ValueError) as err:

            failed.append((str(path), str(err)))

            print(f"[{i}/{len(paths)}] SKIPPED {clip_id}: {err}")

    # Combine new and previously saved results
    if new_landmarks:

        new_df = pd.concat(new_landmarks, ignore_index=True)

        if existing_df is not None:
            raw_df = pd.concat(
                [existing_df, new_df],
                ignore_index=True
            )
        else:
            raw_df = new_df

        raw_df.to_csv(RAW_CSV, index=False)

    elif existing_df is not None:

        raw_df = existing_df

    else:

        raise RuntimeError("No videos were successfully processed.")

    print("\nRAW DATASET")
    print("Shape:", raw_df.shape)
    print("Videos:", raw_df["clip_id"].nunique())
    print("Saved to:", RAW_CSV)
    print("Failed:", len(failed))

    return raw_df, failed


def load_raw_dataset():
    """Load previously extracted landmark coordinates."""

    return pd.read_csv(RAW_CSV)


# ============================================================
# 2. PREPROCESSING
# ============================================================

def df_to_array(df):
    """
    Long format -> (T, 2, 21, 3) array.

    NaN wherever a hand wasn't detected.
    Preserves the original video frame count.
    """

    if df.empty:
        raise ValueError("No landmarks in this clip.")

    first = int(df["frame"].min())

    if "frame_count" in df.columns:
        T = int(df["frame_count"].iloc[0])
    else:
        T = int(df["frame"].max() - first + 1)

    arr = np.full(
        (T, N_HANDS, N_LANDMARKS, 3),
        np.nan
    )

    frame_idx = df["frame"].values - first

    arr[
        frame_idx,
        df["hand"].values,
        df["landmark"].values
    ] = df[["x", "y", "z"]].values

    return arr


def fill_gaps(arr, min_frac=0.3):
    """
    Interpolate short detection gaps.

    A hand seen in fewer than min_frac of frames
    is treated as not used and set to zeros.
    """

    out = arr.copy()
    t = np.arange(len(out))

    for h in range(N_HANDS):

        present = ~np.isnan(out[:, h, 0, 0])

        if present.mean() < min_frac:
            out[:, h] = 0.0
            continue

        for l in range(N_LANDMARKS):

            for c in range(3):

                out[:, h, l, c] = np.interp(
                    t,
                    t[present],
                    out[present, h, l, c]
                )

    return out


def normalize(arr):
    """
    Normalize hand landmarks relative to wrist position
    and hand size.

    Returns:
        (T, 130) array
    """

    T = len(arr)

    # Wrist-relative coordinates
    rel = arr - arr[:, :, 0:1, :]

    # Hand size: wrist to middle-finger base
    size = np.linalg.norm(
        arr[:, :, 9, :2] - arr[:, :, 0, :2],
        axis=-1
    )

    size = np.where(size > 1e-6, size, 1.0)

    rel = rel / size[:, :, None, None]

    # Preserve wrist position
    wrist_pos = arr[:, :, 0, :2]

    return np.concatenate(
        [
            rel.reshape(T, -1),
            wrist_pos.reshape(T, -1)
        ],
        axis=1
    )


def resample(seq, n_frames=30):
    """
    Resample sequence to exactly n_frames.
    """

    old = np.linspace(0, 1, len(seq))
    new = np.linspace(0, 1, n_frames)

    return np.stack(
        [
            np.interp(new, old, seq[:, i])
            for i in range(seq.shape[1])
        ],
        axis=1
    )


# ============================================================
# 3. BASELINE MODEL DATASET
# ============================================================

def clip_to_row(df, n_frames=30, mode="summary", **meta):
    """
    Convert one clip into a single feature row.

    summary: mean, std, min, max (520 features)
    flatten: all frames flattened (n_frames * 130 features)
    """

    seq = resample(
        normalize(fill_gaps(df_to_array(df))),
        n_frames
    )

    if mode == "summary":

        feats = np.concatenate([
            seq.mean(0),
            seq.std(0),
            seq.min(0),
            seq.max(0)
        ])

    elif mode == "flatten":

        feats = seq.ravel()

    else:

        raise ValueError("mode must be 'summary' or 'flatten'")

    feat_df = pd.DataFrame(
        [feats],
        columns=[f"f{i}" for i in range(len(feats))]
    )

    if meta:
        return pd.concat(
            [pd.DataFrame([meta]), feat_df],
            axis=1
        )

    return feat_df


def build_dataset_from_csv(n_frames=30, mode="summary"):
    """
    Build baseline dataset from saved raw landmarks.

    No MediaPipe processing occurs here.
    """

    raw_df = load_raw_dataset()

    rows = []
    failed = []

    grouped = raw_df.groupby("clip_id")

    for clip_id, landmarks in grouped:

        try:

            # Retrieve metadata
            label = landmarks["label"].iloc[0]
            signer = landmarks["signer"].iloc[0]
            path = landmarks["path"].iloc[0]

            rows.append(
                clip_to_row(
                    landmarks,
                    n_frames=n_frames,
                    mode=mode,
                    clip_id=clip_id,
                    label=label,
                    signer=signer,
                    path=path
                )
            )

        except (RuntimeError, ValueError) as err:

            failed.append((clip_id, str(err)))

            continue

    if not rows:
        raise RuntimeError("No clips successfully processed.")

    X_df = pd.concat(rows, ignore_index=True)

    print("\nBASELINE DATASET")
    print("Shape:", X_df.shape)
    print("Features:", X_df.shape[1] - 4)
    print("Failed:", len(failed))

    return X_df, failed


# ============================================================
# 4. LSTM DATASET
# ============================================================

def clip_to_sequence(df, n_frames=30):
    """
    Convert one video into an LSTM-ready sequence.

    Returns:
        (n_frames, 130) array
    """

    seq = df_to_array(df)

    seq = fill_gaps(seq)

    seq = normalize(seq)

    seq = resample(seq, n_frames)

    return seq


def build_lstm_dataset_from_csv(n_frames=30):
    """
    Build LSTM dataset from saved raw landmarks.

    No MediaPipe processing occurs here.

    Returns:
        X: (n_clips, n_frames, 130)
        y: labels
        metadata_df
        failed
    """

    raw_df = load_raw_dataset()

    X = []
    y = []
    metadata = []
    failed = []

    grouped = raw_df.groupby("clip_id")

    for clip_id, landmarks in grouped:

        try:

            label = landmarks["label"].iloc[0]
            signer = landmarks["signer"].iloc[0]
            path = landmarks["path"].iloc[0]

            sequence = clip_to_sequence(
                landmarks,
                n_frames=n_frames
            )

            X.append(sequence)
            y.append(label)

            metadata.append({
                "clip_id": clip_id,
                "label": label,
                "signer": signer,
                "path": path
            })

        except (RuntimeError, ValueError) as err:

            failed.append((clip_id, str(err)))

            continue

    X = np.array(X, dtype=np.float32)
    y = np.array(y)

    metadata_df = pd.DataFrame(metadata)

    print("\nLSTM DATASET")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Failed:", len(failed))

    return X, y, metadata_df, failed
